In [1]:
import os
os.environ["PYSPARK_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col, to_date, to_timestamp, datediff, months_between,
    current_date, current_timestamp, year, month, dayofmonth,
    dayofweek, quarter, date_format, date_add, date_sub,
    round, when, sum, count, lit
)

spark = SparkSession.builder.appName("Week2_Day4").master("local[*]").getOrCreate()

# Employees with hire dates as STRINGS (messy, like real life)
emp_data = [
    (1, "Alice",   "Engineering", 95000, "2019-03-15"),
    (2, "Bob",     "Engineering", 90000, "2020-07-22"),
    (3, "Charlie", "Data",        85000, "2018-11-01"),
    (4, "Diana",   "Data",        80000, "2023-01-10"),
    (5, "Eve",     "HR",          70000, "2021-06-30"),
    (6, "Frank",   "HR",          65000, "2024-09-05"),
]

emp_schema = StructType([
    StructField("emp_id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("department", StringType(), False),
    StructField("salary", IntegerType(), False),
    StructField("hire_date_str", StringType(), False),
])

df_emp = spark.createDataFrame(emp_data, emp_schema)

# Orders with timestamps
orders_data = [
    (101, "2024-01-15 09:30:00", 250.0),
    (102, "2024-01-20 14:15:00", 180.0),   # Saturday
    (103, "2024-02-03 11:00:00", 320.0),   # Saturday
    (104, "2024-02-14 16:45:00", 450.0),
    (105, "2024-03-08 08:20:00", 190.0),
    (106, "2024-03-17 13:10:00", 275.0),   # Sunday
    (107, "2024-04-01 10:00:00", 600.0),
    (108, "2024-04-21 15:30:00", 340.0),   # Sunday
    (109, "2024-05-06 09:45:00", 410.0),
    (110, "2024-06-15 12:00:00", 520.0),   # Saturday
]

orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("order_timestamp_str", StringType(), False),
    StructField("amount", DoubleType(), False),
])

df_orders = spark.createDataFrame(orders_data, orders_schema)

In [4]:
df_emp_clean = df_emp.withColumn(
    "hire_date",
    to_date(col("hire_date_str"), "yyyy-MM-dd")
)

df_emp_clean.select("name", "hire_date_str", "hire_date").show()

+-------+-------------+----------+
|   name|hire_date_str| hire_date|
+-------+-------------+----------+
|  Alice|   2019-03-15|2019-03-15|
|    Bob|   2020-07-22|2020-07-22|
|Charlie|   2018-11-01|2018-11-01|
|  Diana|   2023-01-10|2023-01-10|
|    Eve|   2021-06-30|2021-06-30|
|  Frank|   2024-09-05|2024-09-05|
+-------+-------------+----------+



In [8]:
# Different formats you'll encounter
messy_dates = [
    (1, "15/03/2024"),      # dd/MM/yyyy
    (2, "03-15-2024"),      # MM-dd-yyyy
    (3, "20240315"),        # yyyyMMdd
]

df_messy = spark.createDataFrame(messy_dates, ["id", "date_str"])

df_parsed =df_messy.withColumn(
    "parsed_v1",to_date(col("date_str"), "dd/MM/yyyy") 
    ).withColumn(
    "parsed_v2", to_date(col("date_str"), "MM-dd-yyyy")
    ).withColumn(
    "parsed_v3", to_date(col("date_str"), "yyyyMMdd")
    )

df_parsed.show()

+---+----------+----------+----------+----------+
| id|  date_str| parsed_v1| parsed_v2| parsed_v3|
+---+----------+----------+----------+----------+
|  1|15/03/2024|2024-03-15|      NULL|      NULL|
|  2|03-15-2024|      NULL|2024-03-15|      NULL|
|  3|  20240315|      NULL|      NULL|2024-03-15|
+---+----------+----------+----------+----------+



In [9]:
df_orders_clean = df_orders.withColumn(
    "order_timestamp",
    to_timestamp(col("order_timestamp_str"), "yyyy-MM-dd HH:mm:ss")
).withColumn(
    "order_date",
    to_date(col("order_timestamp_str"), "yyyy-MM-dd HH:mm:ss")
)

df_orders_clean.show()

+--------+-------------------+------+-------------------+----------+
|order_id|order_timestamp_str|amount|    order_timestamp|order_date|
+--------+-------------------+------+-------------------+----------+
|     101|2024-01-15 09:30:00| 250.0|2024-01-15 09:30:00|2024-01-15|
|     102|2024-01-20 14:15:00| 180.0|2024-01-20 14:15:00|2024-01-20|
|     103|2024-02-03 11:00:00| 320.0|2024-02-03 11:00:00|2024-02-03|
|     104|2024-02-14 16:45:00| 450.0|2024-02-14 16:45:00|2024-02-14|
|     105|2024-03-08 08:20:00| 190.0|2024-03-08 08:20:00|2024-03-08|
|     106|2024-03-17 13:10:00| 275.0|2024-03-17 13:10:00|2024-03-17|
|     107|2024-04-01 10:00:00| 600.0|2024-04-01 10:00:00|2024-04-01|
|     108|2024-04-21 15:30:00| 340.0|2024-04-21 15:30:00|2024-04-21|
|     109|2024-05-06 09:45:00| 410.0|2024-05-06 09:45:00|2024-05-06|
|     110|2024-06-15 12:00:00| 520.0|2024-06-15 12:00:00|2024-06-15|
+--------+-------------------+------+-------------------+----------+



In [10]:
df_emp_clean.select(
    "name",
    current_date().alias("today"),
    current_timestamp().alias("right_now")
).show(3, truncate=False)

+-------+----------+--------------------------+
|name   |today     |right_now                 |
+-------+----------+--------------------------+
|Alice  |2026-03-14|2026-03-14 17:02:01.151937|
|Bob    |2026-03-14|2026-03-14 17:02:01.151937|
|Charlie|2026-03-14|2026-03-14 17:02:01.151937|
+-------+----------+--------------------------+
only showing top 3 rows



In [17]:
import pyspark.sql.functions as F
df_parts = df_emp_clean.select(
    "name",
    "hire_date",
    year("hire_date").alias("hire_year"),
    month("hire_date").alias("hire_month"),
    dayofmonth("hire_date").alias("hire_day"),
    quarter("hire_date").alias("hire_quarter"),
    dayofweek("hire_date").alias("hire_dow")
)

df_parts.show()

+-------+----------+---------+----------+--------+------------+--------+
|   name| hire_date|hire_year|hire_month|hire_day|hire_quarter|hire_dow|
+-------+----------+---------+----------+--------+------------+--------+
|  Alice|2019-03-15|     2019|         3|      15|           1|       6|
|    Bob|2020-07-22|     2020|         7|      22|           3|       4|
|Charlie|2018-11-01|     2018|        11|       1|           4|       5|
|  Diana|2023-01-10|     2023|         1|      10|           1|       3|
|    Eve|2021-06-30|     2021|         6|      30|           2|       4|
|  Frank|2024-09-05|     2024|         9|       5|           3|       5|
+-------+----------+---------+----------+--------+------------+--------+



In [19]:
df_tenure = df_emp_clean.withColumn(
    "days_at_company",
    datediff(current_date(),col("hire_date"))
)

df_tenure.select("name","hire_date","days_at_company").show()

+-------+----------+---------------+
|   name| hire_date|days_at_company|
+-------+----------+---------------+
|  Alice|2019-03-15|           2556|
|    Bob|2020-07-22|           2061|
|Charlie|2018-11-01|           2690|
|  Diana|2023-01-10|           1159|
|    Eve|2021-06-30|           1718|
|  Frank|2024-09-05|            555|
+-------+----------+---------------+



In [20]:
df_tenure2 = df_emp_clean.withColumn(
    "months_at_cop",
    round(months_between(current_date(), col("hire_date")),1)
).withColumn(
    "years_at_cop",
    round(months_between(current_date(),col("hire_date"))/12,1)
)
df_tenure2.show()

+------+-------+-----------+------+-------------+----------+-------------+------------+
|emp_id|   name| department|salary|hire_date_str| hire_date|months_at_cop|years_at_cop|
+------+-------+-----------+------+-------------+----------+-------------+------------+
|     1|  Alice|Engineering| 95000|   2019-03-15|2019-03-15|         84.0|         7.0|
|     2|    Bob|Engineering| 90000|   2020-07-22|2020-07-22|         67.7|         5.6|
|     3|Charlie|       Data| 85000|   2018-11-01|2018-11-01|         88.4|         7.4|
|     4|  Diana|       Data| 80000|   2023-01-10|2023-01-10|         38.1|         3.2|
|     5|    Eve|         HR| 70000|   2021-06-30|2021-06-30|         56.5|         4.7|
|     6|  Frank|         HR| 65000|   2024-09-05|2024-09-05|         18.3|         1.5|
+------+-------+-----------+------+-------------+----------+-------------+------------+



In [23]:
df_tenure2.filter(col("years_at_cop") > 5).select("name", "years_at_cop").show()

+-------+------------+
|   name|years_at_cop|
+-------+------------+
|  Alice|         7.0|
|    Bob|         5.6|
|Charlie|         7.4|
+-------+------------+



In [26]:
df_emp_clean.select(
    "name",
    "hire_date",
    date_add(col("hire_date"),90).alias("after_propabtion"),
    date_sub(col("hire_date"),14).alias("offer_accepted"),
    
).show()

+-------+----------+----------------+--------------+
|   name| hire_date|after_propabtion|offer_accepted|
+-------+----------+----------------+--------------+
|  Alice|2019-03-15|      2019-06-13|    2019-03-01|
|    Bob|2020-07-22|      2020-10-20|    2020-07-08|
|Charlie|2018-11-01|      2019-01-30|    2018-10-18|
|  Diana|2023-01-10|      2023-04-10|    2022-12-27|
|    Eve|2021-06-30|      2021-09-28|    2021-06-16|
|  Frank|2024-09-05|      2024-12-04|    2024-08-22|
+-------+----------+----------------+--------------+



In [28]:
df_emp_clean.select(
    "name",
    "hire_date",
    date_format(col("hire_date"), "dd/MM/yyyy").alias("european_format"),
    date_format(col("hire_date"), "MMMM dd, yyyy").alias("readable_format"),
    date_format(col("hire_date"), "yyyy-'Q'Q").alias("Year_quater"),
    date_format(col("hire_date"), "EEEE").alias("day_name"),
).show()

+-------+----------+---------------+------------------+-----------+---------+
|   name| hire_date|european_format|   readable_format|Year_quater| day_name|
+-------+----------+---------------+------------------+-----------+---------+
|  Alice|2019-03-15|     15/03/2019|    March 15, 2019|    2019-Q1|   Friday|
|    Bob|2020-07-22|     22/07/2020|     July 22, 2020|    2020-Q3|Wednesday|
|Charlie|2018-11-01|     01/11/2018| November 01, 2018|    2018-Q4| Thursday|
|  Diana|2023-01-10|     10/01/2023|  January 10, 2023|    2023-Q1|  Tuesday|
|    Eve|2021-06-30|     30/06/2021|     June 30, 2021|    2021-Q2|Wednesday|
|  Frank|2024-09-05|     05/09/2024|September 05, 2024|    2024-Q3| Thursday|
+-------+----------+---------------+------------------+-----------+---------+



In [29]:
df_orders_with_dates = df_orders.withColumn(
    "order_date",
    to_date(col("order_timestamp_str"), "yyyy-MM-dd HH:mm:ss")
)

df_quarterly = df_orders_with_dates.withColumn(
    "quarter", quarter("order_date")
).withColumn(
    "year", year("order_date")
).groupBy("year", "quarter").agg(
    sum("amount").alias("quarterly_revenue"),
    count("order_id").alias("order_count")
).orderBy("year", "quarter")

df_quarterly.show()

+----+-------+-----------------+-----------+
|year|quarter|quarterly_revenue|order_count|
+----+-------+-----------------+-----------+
|2024|      1|           1665.0|          6|
|2024|      2|           1870.0|          4|
+----+-------+-----------------+-----------+



In [30]:
df_weekday = df_orders_with_dates.withColumn(
    "dow", dayofweek("order_date")
).withColumn(
    "day_type",
    when(col("dow").isin(1,7), "weekend").otherwise("weekday")
)

# Check individual orders
df_weekday.select("order_id", "order_date", "dow", "day_type", "amount").show()

+--------+----------+---+--------+------+
|order_id|order_date|dow|day_type|amount|
+--------+----------+---+--------+------+
|     101|2024-01-15|  2| weekday| 250.0|
|     102|2024-01-20|  7| weekend| 180.0|
|     103|2024-02-03|  7| weekend| 320.0|
|     104|2024-02-14|  4| weekday| 450.0|
|     105|2024-03-08|  6| weekday| 190.0|
|     106|2024-03-17|  1| weekend| 275.0|
|     107|2024-04-01|  2| weekday| 600.0|
|     108|2024-04-21|  1| weekend| 340.0|
|     109|2024-05-06|  2| weekday| 410.0|
|     110|2024-06-15|  7| weekend| 520.0|
+--------+----------+---+--------+------+



In [33]:
from pyspark.sql.functions import avg
df_weekday.groupby("day_type").agg(
    count("order_id").alias("total_orders"),
    round(sum("amount"), 2).alias("total_revenue"),
    round(avg("amount"), 2).alias("avg_order_value")
).show()

+--------+------------+-------------+---------------+
|day_type|total_orders|total_revenue|avg_order_value|
+--------+------------+-------------+---------------+
| weekday|           5|       1900.0|          380.0|
| weekend|           5|       1635.0|          327.0|
+--------+------------+-------------+---------------+

